In [ ]:
import sympy as sp
import numpy as np
from numpy import sqrt, kron, trace
from functools import reduce
import pandas as pd

q = sp.Symbol("q")
lam = sp.Symbol("lambda")
p_r = sp.Symbol("p_r")
p_l = sp.Symbol("p_l")

H = np.array([[1 / sqrt(2), 1 / sqrt(2)], [1 / sqrt(2), -1 / sqrt(2)]])
H3 = np.kron(np.kron(H, H), H)

def kron_all(*ops):
    return reduce(kron, ops)


def normalize(dm):
    return dm / trace(dm)

identity = np.array([[1, 0], [0, 1]])
x_gate = np.array([[0, 1], [1, 0]])
y_gate = np.array([[0, -1j], [1j, 0]])
z_gate = np.array([[1, 0], [0, -1]])
pauli_gates = [identity, x_gate, y_gate, z_gate]

proj_0 = np.array([[1, 0], [0, 0]])
proj_1 = np.array([[0, 0], [0, 1]])
ket_0 = np.array([[1], [0]])
ket_1 = np.array([[0], [1]])

U_03 = kron_all(proj_0, identity, identity, identity, identity, identity) + kron_all(
    proj_1, identity, identity, x_gate, identity, identity
)
U_14 = kron_all(identity, proj_0, identity, identity, identity, identity) + kron_all(
    identity, proj_1, identity, identity, x_gate, identity
)
U_25 = kron_all(identity, identity, proj_0, identity, identity, identity) + kron_all(
    identity, identity, proj_1, identity, identity, x_gate
)

ket_abc000 = kron_all(identity, identity, identity, ket_0, ket_0, ket_0)
ket_abc111 = kron_all(identity, identity, identity, ket_1, ket_1, ket_1)


def apply_AD(dm):
    input_dm = kron(dm, dm)

    # rotate
    input_dm = kron(H3, H3) @ input_dm @ kron(H3, H3)
    output_dm = U_03 @ (U_14 @ (U_25 @ input_dm @ U_25.T) @ U_14.T) @ U_03.T
    output_dm = (ket_abc000.T @ output_dm @ ket_abc000) + (
        ket_abc111.T @ output_dm @ ket_abc111
    )
    prob_suc = trace(output_dm)

    # rotate back
    output_dm = H3 @ output_dm @ H3
    return normalize(output_dm), prob_suc

def get_phase_QBER(dm):
    """
    Extract the phase error rate for the W-state protocol given a density matrix.
    The density matrix should be in the COMPUTATIONAL basis.
    """

    # ZZZ = 1
    # Components p_000, p_011, p_101, p_110
    ZZZ_p = dm[0, 0] + dm[3, 3] + dm[5, 5] + dm[6, 6]

    # ZZZ = -1
    # Components p_111, p_001, p_010, p_100
    ZZZ_m = dm[7, 7] + dm[1, 1] + dm[2, 2] + dm[4, 4]

    ZZZ_exp = ZZZ_p - ZZZ_m

    QBER = (1 + ZZZ_exp) / 2

    return QBER


def get_bit_QBER(dm):
    """
    Extract the bit error rate of the W-state protocol given a density matrix.
    The density matrix should be in the COMPUTATIONAL basis.
    """
    rotated_dm = H3 @ dm @ H3
    # X_1 X_2 = 1 (Alice and Bob are the same)
    # Components p_000, p_001, p_110, p_111

    XX_AB_p = rotated_dm[0, 0] + rotated_dm[1, 1] + rotated_dm[6, 6] + rotated_dm[7, 7]

    # X_1 X_2 = -1 (Alice and Bob are different)
    XX_AB_m = 1 - XX_AB_p

    # X_1 X_3 = 1 (Alice and Charlie are the same)
    # Components p_000, p_101, p_010, p_111

    XX_AC_p = rotated_dm[0, 0] + rotated_dm[5, 5] + rotated_dm[2, 2] + rotated_dm[7, 7]

    # X_1 X_3 = -1 (Alice and Charlie are different)
    XX_AC_m = 1 - XX_AC_p

    XX_AB_exp = XX_AB_p - XX_AB_m
    XX_AC_exp = XX_AC_p - XX_AC_m

    QBER_AB = float((1 - XX_AB_exp) / 2)
    QBER_AC = float((1 - XX_AC_exp) / 2)

    return max(QBER_AB, QBER_AC)


def get_bin_entropy(p):
    """
    Return the binary entropy.
    """
    if p == 0 or p == 1 or p < 0:
        return 0
    else:
        return (-p * (np.log2(p))) + (-(1 - p) * np.log2(1 - p))


def get_sk_fraction(dm):
    bit_QBER = get_bit_QBER(dm)
    phase_QBER = get_phase_QBER(dm)

    return max(1 - get_bin_entropy(phase_QBER) - get_bin_entropy(bit_QBER), 0)


def get_sk_fraction_AD(dm):
    bit_QBER = get_bit_QBER(dm)
    phase_QBER = get_phase_QBER(dm)

    return max(1 - get_bin_entropy(phase_QBER) - get_bin_entropy(bit_QBER), 0) / 2


def get_single_pauli(qubit_idx, pauli_idx):
    result = [identity, identity, identity]
    result[qubit_idx] = pauli_gates[pauli_idx]
    return kron_all(*result)


def apply_depolarization(dm, dep):
    result = dm
    x_gate_0 = get_single_pauli(0, 1)
    y_gate_0 = get_single_pauli(0, 2)
    z_gate_0 = get_single_pauli(0, 3)

    x_gate_1 = get_single_pauli(1, 1)
    y_gate_1 = get_single_pauli(1, 2)
    z_gate_1 = get_single_pauli(1, 3)

    x_gate_2 = get_single_pauli(2, 1)
    y_gate_2 = get_single_pauli(2, 2)
    z_gate_2 = get_single_pauli(2, 3)

    dm_noise = (dep / 3) * (
        (x_gate_0 @ dm @ x_gate_0.conj().T)
        + (y_gate_0 @ dm @ y_gate_0.conj().T)
        + (z_gate_0 @ dm @ z_gate_0.conj().T)
    )
    result = (1 - dep) * dm + dm_noise

    dm_noise = (dep / 3) * (
        (x_gate_1 @ result @ x_gate_1.conj().T)
        + (y_gate_1 @ result @ y_gate_1.conj().T)
        + (z_gate_1 @ result @ z_gate_1.conj().T)
    )
    result = (1 - dep) * result + dm_noise

    dm_noise = (dep / 3) * (
        (x_gate_2 @ result @ x_gate_2.conj().T)
        + (y_gate_2 @ result @ y_gate_2.conj().T)
        + (z_gate_2 @ result @ z_gate_2.conj().T)
    )
    result = (1 - dep) * result + dm_noise

    return result

W_ket = (1 / sqrt(3)) * (
    kron_all(ket_0, ket_0, ket_1)
    + kron_all(ket_0, ket_1, ket_0)
    + kron_all(ket_1, ket_0, ket_0)
)
W_dm = W_ket @ W_ket.T
dep_lst = np.linspace(0, 0.1, 20)
sk_raw_lst = []
sk_AD_lst = []
for dep in dep_lst:
    dm_raw = apply_depolarization(W_dm, dep)
    dm_AD, prob_AD = apply_AD(dm_raw)
    sk_raw = get_sk_fraction(dm_raw)
    sk_AD = get_sk_fraction_AD(dm_AD) * prob_AD
    sk_raw_lst.append(np.real(sk_raw))
    sk_AD_lst.append(np.real(sk_AD))
df = pd.DataFrame(
    {"dep_lst": dep_lst, "sk_raw_lst": sk_raw_lst, "sk_AD_lst": sk_AD_lst}
)
df.to_csv("depolarization_AD.csv", index=False, sep=",")